# Phase 0 — EDA, shared evaluation, normalisation

**Business Entity Resolution (Amazon ML Challenge 2026)**

| Task | What it delivers |
|---|---|
| T1 EDA | the facts about the data that drive every design decision |
| T0 Shared evaluation | fixed holdout + exact macro-F0.5 scorer + candidate-set report, used by the whole team |
| T2 Normalisation | one library that turns every raw record into clean name / address fields |
| Baseline | first end-to-end number on the holdout, so every later idea has something to beat |

Code lives in `../src/`. Data is read from `../../dataset/` (override with `BER_DATA_DIR`).

In [ ]:
import os, sys, re, time, json
import numpy as np, pandas as pd
sys.path.insert(0, os.path.abspath("../src"))
pd.set_option("display.width", 220, "display.max_colwidth", 80, "display.max_columns", 30)

from config import DATA_DIR, WORK_DIR, OUT_DIR
from io_utils import read_source, read_gt_pairs, read_tsv
import evaluation as ev
print("DATA_DIR:", DATA_DIR, "| WORK_DIR:", WORK_DIR)
for split in ("train", "test"):
    print(split, sorted(os.listdir(os.path.join(DATA_DIR, split))))

## 1. EDA (T1)

### 1.1 Size and country mix
All files load losslessly with a pyarrow reader (`quote_char=False` because names contain quotes,
empty strings kept as `''`). A 5M-row file loads in ~1-2 s and ~1.4 GB.

In [ ]:
rows = []
for split in ("train", "test"):
    for k in (1, 2, 3):
        d = read_source(split, k)
        vc = d.country.value_counts(normalize=True)
        rows.append({"file": f"{split}_source{k}", "rows": len(d), "unique_ids": d.entity_id.nunique(),
                     **{f"% {c}": round(100 * v, 1) for c, v in vc.items()},
                     "% empty address": round(100 * (d.business_address.str.strip() == "").mean(), 1)})
        del d
overview = pd.DataFrame(rows).fillna(0)
overview

### 1.2 Ground truth: how many matches, exclusivity, countries

In [ ]:
gt_pairs = read_gt_pairs()                                  # one row per (s1_id, rid)
s1_train = read_source("train", 1)
n_match = gt_pairs.groupby("s1_id").size().reindex(s1_train.entity_id, fill_value=0)
print(f"true pairs: {len(gt_pairs):,}")
print(f"singletons (no match): {(n_match == 0).mean():.2%}   mean matches per S1: {n_match.mean():.2f}   max: {n_match.max()}")
display(n_match.clip(upper=9).value_counts(normalize=True).sort_index().rename("share of S1 entities").to_frame().T.round(3))

print("EXCLUSIVITY - each S2/S3 id matched to at most one S1:", gt_pairs.rid.is_unique)
n2, n3 = overview.set_index("file").rows[["train_source2", "train_source3"]]
print(f"share of S2 records that match some S1: {gt_pairs.rid.str.startswith('S2').sum()/n2:.1%}; "
      f"S3: {gt_pairs.rid.str.startswith('S3').sum()/n3:.1%}  -> the rest are distractors")

In [ ]:
# country consistency of true pairs
cty = pd.concat([read_source("train", k, ["entity_id", "country"]) for k in (1, 2, 3)]).set_index("entity_id").country.astype("category")
same = (cty.reindex(gt_pairs.s1_id).to_numpy() == cty.reindex(gt_pairs.rid).to_numpy()).mean()
print(f"true pairs with the same country label: {same:.4%}  -> blocking within country is safe")
del cty; import gc; _ = gc.collect()

### 1.3 Names are not unique (chains / generic names)

In [ ]:
from normalize import normalize_name
_p = os.path.join(WORK_DIR, "norm_train_s1.parquet")
if os.path.exists(_p):          # cached by build_normalized.py (section 3.4)
    s1_core = pd.read_parquet(_p, columns=["n_core"]).n_core.to_numpy()
else:
    s1_core = s1_train.business_name.map(lambda x: normalize_name(x)["n_core"]).to_numpy()
s1_core = pd.Series(s1_core, index=s1_train.index)
for c in ("US", "India"):
    vc = s1_core[s1_train.country == c].value_counts()
    print(f"{c}: S1 entities sharing their core name with another S1 entity: {vc[vc>1].sum()/vc.sum():.1%}"
          f"  | top: {dict(vc.head(6))}")
del s1_core; _ = gc.collect()

### 1.4 Noise patterns per source and country (200k sample each)

In [ ]:
INDIC = "[\u0900-\u0DFF]"
def pattern_stats(df):
    nm, ad = df.business_name, df.business_address
    return {
        "name: Indic script": nm.str.contains(INDIC).mean(),
        "name: domain (.com)": nm.str.contains(r"\.c[o0]m\b", case=False).mean(),
        "name: junk prefix": nm.str.contains(r"^(?:\.\.\.|\*\*\*|>>|--|@|\[|<<)").mean(),
        "name: DBA/aka/t-a": nm.str.contains(r"\b(?:dba|d/b/a|aka|t/a)\b", case=False).mean(),
        "name: ALL CAPS": (nm == nm.str.upper()).mean(),
        "addr: ALL CAPS": (ad == ad.str.upper()).mean(),
        "addr: Indic script": ad.str.contains(INDIC).mean(),
        "addr: empty / null": (ad.str.strip().isin(["", "NULL", "<NULL>", "N/A"])).mean(),
        "addr: 5-digit zip": ad.str.contains(r"(?<!\d)\d{5}(?!\d)").mean(),
        "addr: 6-digit PIN": ad.str.contains(r"(?<!\d)\d{6}(?!\d)").mean(),
    }
rows = {}
for split in ("train", "test"):
    for k in (1, 2, 3):
        d = read_source(split, k)
        for c, g in d.groupby("country"):
            rows[f"{split} S{k} {c}"] = pattern_stats(g.sample(min(len(g), 200_000), random_state=0))
        del d
(pd.DataFrame(rows).T * 100).round(1)

### 1.5 What true matches look like

In [ ]:
rng = np.random.default_rng(3)
samp_ids = set(rng.choice(gt_pairs.s1_id.unique(), 20_000, replace=False))   # also used in section 3.3
need = set(gt_pairs.rid[gt_pairs.s1_id.isin(samp_ids)])
recs = []
for k in (2, 3):
    d = read_source("train", k); recs.append(d[d.entity_id.isin(need)]); del d
recs = pd.concat(recs).set_index("entity_id")
s1i = s1_train[s1_train.entity_id.isin(samp_ids)].set_index("entity_id")
ex_s1 = list(samp_ids)[:5]
for e in ex_s1:
    print(f"S1  {s1i.business_name[e]!r:45} | {s1i.business_address[e]}")
    for r in gt_pairs.rid[gt_pairs.s1_id == e]:
        print(f"  {r[:2]} {recs.business_name[r]!r:45} | {recs.business_address[r]}")
    print()

**Key EDA takeaways**

1. 2.2M train / 1.7M test S1 entities; ~10M S2+S3 records per split. Test adds France (15%).
2. Only ~5.6% singletons; ~3.5 matches per entity (1-3 from each of S2 and S3).
3. **Exclusivity holds**: every S2/S3 record belongs to at most one S1 -> one-to-one assignment post-processing.
4. **Country labels agree in 100% of true pairs** -> block within country.
5. ~26% of S2/S3 records are distractors, many are near-copies of real S1 entities (same name, house number off by a digit).
6. Names are highly non-unique (43% US / 54% India share a core name) -> address must carry the decision.
7. India: ~23% of S2 names are in Indic scripts (9 scripts); postal codes are rare everywhere.

## 2. Shared evaluation (T0)

* **Holdout** = train S1 ids with `int(id[3:]) % 10 == 0` (~220k entities). Everyone reports on this.
* **Scorer** = exact challenge metric (per-S1 F0.5, singletons score 1 only when predicted empty, macro average).
* **Candidate report** = pair recall, avg candidates per S1, and the *oracle* F0.5 a perfect matcher would reach on the candidates.
* **Candidate file format** every blocking method writes: parquet/tsv with `s1_id, rid, method, score`.

In [ ]:
hold_ids = ev.holdout_ids()
print(f"holdout S1 entities: {len(hold_ids):,}")
ev.main(["", "export"])

# scorer checks
P = pd.DataFrame({"s1_id": ["S1-1"]*3, "rid": ["S2-00047", "S2-00193", "S3-00812"]})
T = pd.DataFrame({"s1_id": ["S1-1"]*2, "rid": ["S2-00047", "S3-00812"]})
print("PDF example                      :", round(ev.macro_f05(P, T, ["S1-1"]), 3), "(expected 0.714)")
print("+ singleton predicted empty       :", round(ev.macro_f05(P, T, ["S1-1", "S1-2"]), 4), "(expected 0.857)")
P2 = pd.concat([P, pd.DataFrame({"s1_id": ["S1-2"], "rid": ["S2-9"]})])
print("+ singleton predicted wrongly     :", round(ev.macro_f05(P2, T, ["S1-1", "S1-2"]), 4), "(expected 0.357)")
gt_hold = gt_pairs[gt_pairs.s1_id.isin(set(hold_ids))]
print("predict nothing on holdout        :", round(ev.macro_f05(gt_hold.iloc[:0], gt_hold, hold_ids), 4))
print("perfect prediction on holdout     :", ev.macro_f05(gt_hold, gt_hold, hold_ids))

## 3. Normalisation (T2)

### 3.1 Learn the Indic -> Latin token map from training pairs
Indic-script names are word-by-word transliterations of the S1 name, so aligning the tokens of
true pairs gives a dictionary (learned only from the provided training data).

In [ ]:
import learn_translit
learn_translit.main()
m = json.load(open(os.path.join(WORK_DIR, "translit_map.json"), encoding="utf-8"))
print(list(m["token"].items())[:15])

### 3.2 What the normaliser does — before / after

In [ ]:
import importlib, normalize; importlib.reload(normalize)
from normalize import normalize_name, normalize_address
names = ["Co1lins Ventrus Inc", "BACK ALLEY BARBER5HOP", "MACS CONSULTANCY LTD LTD", "Wexiri d/b/a Tech Care Pvt Ltd",
         "silverportfolio.com", "éqgcapital.c0m", "Sahara Electricals  Pvt Ltd | www.saharaele.com", "#IOBHAI",
         "-- Holloway Peak Inc Seafood", "मॉडर्न डिजिटल कंसल्टेंसी प्राइवेट लिमिटेड",
         "బాలాజీ కన్\u200cస్ట్రక్షన్స్ ప్రైవేట్ లిమిటెడ్", "Capital Academy L.L.C.", "Pvt. EFS Print Ventures Ltd.",
         "BELSUNCE ECOLE S.A.S", "Ets Handi Groupe EI"]
pd.DataFrame([{"raw": n, **{k: v for k, v in normalize_name(n).items() if k in ("n_core", "n_alt", "legal", "is_domain", "was_indic")}} for n in names])

In [ ]:
addrs = [("NC, 890 Silver Fox Drive, Hope Mills", "US"), ("890 1/2 SILVER FOX DRIVE, HOE MILLS, NC", "US"),
         ("00622 Grand Ave, Wisconsin, City Of Superior", "US"), ("AZ, EL MIRAGE, #12306 117TH AVE", "US"),
         ("127TH SAINT, CLEVELAND, OH", "US"),
         ("B-1005, Tower-B, Bulland Heights Crossings Republik, Ghaziabad, Uttar Pradesh", "India"),
         ("A-504 RUPELHEIGHTSMARTAM COLONY, MUMBAI, महाराष्ट्र", "India"), ("Floor-3, Mumbai, Mumbai City, MH", "India"),
         ("63 R. DE DIEPPE, LILLE, Hauts-de-France", "France"), ("NO 10 AV DE JOSSELIN, ST.-NAZAIRE, Pays de la Loire", "France"),
         ("15 R Des Bordes, La Teste-de-buch, Gironde", "France")]
pd.DataFrame([{"raw": a, **{k: v for k, v in normalize_address(a, c).items() if k in ("a_num", "a_street", "a_city", "a_state", "a_nums")}} for a, c in addrs])

### 3.3 Does it help? Similarity of true pairs, naive cleaning vs normaliser (sample of 20k S1 entities)

In [ ]:
from rapidfuzz import fuzz
from unidecode import unidecode
LEG = set("pvt private ltd limited llc llp inc incorporated corp corporation co company the and of".split())
def naive(s):
    s = re.sub(r"[^a-z0-9 ]+", " ", unidecode(s or "").lower().replace("&", " and "))
    return " ".join(w for w in s.split() if w not in LEG)

sp = gt_pairs[gt_pairs.s1_id.isin(samp_ids)].copy()
sp["n1"] = s1i.business_name.reindex(sp.s1_id).to_numpy(); sp["nr"] = recs.business_name.reindex(sp.rid).to_numpy()
sp["a1"] = s1i.business_address.reindex(sp.s1_id).to_numpy(); sp["ar"] = recs.business_address.reindex(sp.rid).to_numpy()
sp["country"] = s1i.country.reindex(sp.s1_id).to_numpy(); sp["src"] = sp.rid.str.slice(0, 2)
sp["indic"] = sp.nr.str.contains(INDIC)
N1 = [normalize_name(x) for x in sp.n1]; NR = [normalize_name(x) for x in sp.nr]
A1 = [normalize_address(x, c) for x, c in zip(sp.a1, sp.country)]; AR = [normalize_address(x, c) for x, c in zip(sp.ar, sp.country)]
sp["naive_score"] = [fuzz.token_set_ratio(naive(a), naive(b)) for a, b in zip(sp.n1, sp.nr)]
sp["new_score"] = [max(fuzz.token_set_ratio(x["n_core"], y["n_core"]), fuzz.ratio(x["n_nospace"], y["n_nospace"]),
                       fuzz.token_set_ratio(x["n_core"], y["n_alt"]) if y["n_alt"] else 0) for x, y in zip(N1, NR)]
sp["naive_exact"] = [naive(a) == naive(b) for a, b in zip(sp.n1, sp.nr)]
sp["new_exact"] = [x["n_core"] == y["n_core"] for x, y in zip(N1, NR)]
sp["house_no_match"] = [bool(x["a_num"]) and x["a_num"].lstrip("0") == y["a_num"].lstrip("0") for x, y in zip(A1, AR)]
sp["city_match"] = [bool(x["a_city"]) and (x["a_city"] == y["a_city"] or x["a_city"] in y["a_locs"].split("|")) for x, y in zip(A1, AR)]
sp["state_match"] = [bool(x["a_state"]) and x["a_state"] == y["a_state"] for x, y in zip(A1, AR)]
display(sp.groupby(["country", "src"])[["naive_exact", "new_exact", "naive_score", "new_score", "house_no_match", "city_match", "state_match"]].mean().round(3))
display(sp[sp.indic].groupby("src")[["naive_score", "new_score"]].mean().round(1).rename(index=lambda s: s + " (Indic-script names)"))
print(f"true pairs with name score < 60: naive {np.mean(sp.naive_score < 60):.2%} -> normaliser {np.mean(sp.new_score < 60):.2%}")

### 3.4 Normalise every record once (cached parquet used by all later phases)
~70 µs/record -> about 30 CPU-minutes for all 26M records (a few minutes on an 8-16 core machine).
Skips files that already exist.

In [ ]:
import build_normalized
for split in ("train", "test"):
    for k in (1, 2, 3):
        p = os.path.join(WORK_DIR, f"norm_{split}_s{k}.parquet")
        if not os.path.exists(p):
            build_normalized.build(split, k)
        print(p, f"{os.path.getsize(p)/1e6:.0f} MB")
import pyarrow.parquet as pq
del sp, N1, NR, A1, AR, recs; gc.collect()
pq.ParquetFile(os.path.join(WORK_DIR, "norm_test_s2.parquet")).read_row_group(0).to_pandas().sample(8, random_state=1)

## 4. Baseline: how far do exact keys get us?

A deliberately simple reference point on the holdout:
* **Blocking**: same country AND (same core name OR same house number + city).
* **Matcher**: accept a candidate when core name is equal AND (city OR house number is equal), then keep each S2/S3 record only for its best S1.

Every later phase has to beat these numbers.

In [ ]:
del s1_train; gc.collect()

def load_norm(split, k, cols):
    return pd.read_parquet(os.path.join(WORK_DIR, f"norm_{split}_s{k}.parquet"), columns=cols)

def h(*cols):                       # 64-bit hash of concatenated string columns -> compact join keys
    s = cols[0].astype(str)
    for c in cols[1:]:
        s = s + "|" + c.astype(str)
    return pd.util.hash_pandas_object(s, index=False).to_numpy()

def encode_id(ids):                 # 'S2-123' -> 2_000_000_000_123 (int64) to keep joins light
    ids = pd.Series(ids)
    return ids.str.slice(1, 2).astype("int64").to_numpy() * 10**12 + ids.str.slice(3).astype("int64").to_numpy()

def decode_id(x):
    x = np.asarray(x)
    return pd.Series([f"S{a}-{b}" for a, b in zip(x // 10**12, x % 10**12)], dtype="string[pyarrow]")

def keyed(split, k, holdout_only=False):
    d = load_norm(split, k, ["entity_id", "country", "n_core", "a_num", "a_city"])
    if holdout_only:
        d = d[ev.is_holdout(d.entity_id)]
    num = d.a_num.str.lstrip("0")
    out = pd.DataFrame({"id": encode_id(d.entity_id), "k_name": h(d.country, d.n_core),
                        "k_city": h(d.country, d.a_city), "k_num": h(d.country, num),
                        "has_num": (num != "").to_numpy(), "has_city": (d.a_city != "").to_numpy()})
    del d; gc.collect()
    return out

q = keyed("train", 1, holdout_only=True)
r = pd.concat([keyed("train", 2), keyed("train", 3)], ignore_index=True)
print(f"holdout S1: {len(q):,}   S2+S3 pool: {len(r):,}")

# block A: same country + identical core name (skip generic-name blocks with > 100 records)
cnt = r.k_name.map(r.k_name.value_counts())
cA = q[["id", "k_name"]].merge(r.loc[cnt <= 100, ["id", "k_name"]], on="k_name", suffixes=("", "_r"))
# block B: same country + same house number + same city (skip keys with > 50 records, e.g. 'no. 1, Mumbai')
rb = r.loc[r.has_num & r.has_city, ["id", "k_num", "k_city"]]
rb = rb[rb.groupby(["k_num", "k_city"]).id.transform("size") <= 50]
cB = q.loc[q.has_num & q.has_city, ["id", "k_num", "k_city"]].merge(rb, on=["k_num", "k_city"], suffixes=("", "_r"))
del rb, cnt; gc.collect()
cands_int = pd.concat([cA[["id", "id_r"]].assign(method="exact_core_name"),
                       cB[["id", "id_r"]].assign(method="house_no+city")], ignore_index=True)
cands = pd.DataFrame({"s1_id": decode_id(cands_int.id), "rid": decode_id(cands_int.id_r), "method": cands_int.method.to_numpy()})
rep = ev.candidate_report(cands, gt_hold, hold_ids)
by = rep.pop("by_method"); display(pd.Series(rep).to_frame("baseline blocking")); display(pd.DataFrame(by).T)

In [ ]:
# simple rule matcher on those candidates: accept if (name equal AND (city OR house no equal)),
# then keep each S2/S3 record only for its best S1 (exclusivity)
cu = cands_int[["id", "id_r"]].drop_duplicates()
qk = q.set_index("id"); rk = r.set_index("id")
eq = lambda col: qk[col].reindex(cu.id).to_numpy() == rk[col].reindex(cu.id_r).to_numpy()
cu = cu.assign(name_eq=eq("k_name"), city_eq=eq("k_city") & qk.has_city.reindex(cu.id).to_numpy(),
               num_eq=eq("k_num") & qk.has_num.reindex(cu.id).to_numpy())
cu["score"] = cu.name_eq * 2 + cu.city_eq + cu.num_eq * 2
pred = cu[cu.name_eq & (cu.city_eq | cu.num_eq)].sort_values("score", ascending=False).drop_duplicates("id_r")
pred = pd.DataFrame({"s1_id": decode_id(pred.id), "rid": decode_id(pred.id_r)})
f = ev.macro_f05(pred, gt_hold, hold_ids)
print(f"BASELINE holdout macro F0.5 = {f:.4f}   (predicted pairs: {len(pred):,}, true pairs: {len(gt_hold):,})")

## 5. What Phase 0 hands to the team

* `src/evaluation.py` — `python src/evaluation.py candidates <your_candidates.parquet>` prints recall / size / oracle F0.5 on the shared holdout.
* `src/normalize.py` — `normalize_name`, `normalize_address`, `normalize_df`; cached outputs in `work/norm_*.parquet`.
* `work/translit_map.json` — learned Indic -> Latin map.
* `work/holdout_s1_ids.tsv` — the shared holdout ids.